# 10 — Commercial Due Diligence
> Feed in a set of company documents — financials, contracts, regulatory notices, management bios — and get back a scored risk register with a clear go/no-go recommendation, ready for a deal committee.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI


# --- Data shapes ---

class DocumentFindings(BaseModel):
    document_name: str
    document_type: Literal[
        "financials", "contract", "management_bio", "regulatory", "corporate", "other"
    ]
    key_findings: List[str] = Field(
        description="Concrete facts extracted from this document, each as one sentence"
    )
    red_flags: List[str] = Field(
        description="Issues in this document that warrant closer scrutiny"
    )
    questions_raised: List[str] = Field(
        description="Questions this document raises that need to be answered"
    )


class RiskItem(BaseModel):
    area: Literal[
        "financial", "commercial", "operational", "legal", "management", "regulatory"
    ]
    severity: Literal["critical", "high", "medium", "low"]
    likelihood: Literal["high", "medium", "low"]
    title: str = Field(description="Short title for this risk, e.g. 'Revenue concentration'")
    finding: str = Field(
        description="Specific evidence from the documents that supports this risk"
    )
    source_document: str = Field(
        description="Which document(s) this finding came from"
    )
    mitigation: str = Field(
        description="Recommended mitigation or further investigation required"
    )


class DDReport(BaseModel):
    target_company: Optional[str] = Field(
        default=None, description="Name of the company being reviewed if identifiable"
    )
    documents_reviewed: List[str] = Field(
        description="List of documents that were analysed"
    )
    overall_assessment: Literal["proceed", "proceed_with_conditions", "do_not_proceed"]
    executive_summary: str = Field(
        description="3-4 sentence summary for a deal committee"
    )
    risk_items: List[RiskItem] = Field(
        description="All identified risks sorted by severity then likelihood"
    )
    key_conditions: List[str] = Field(
        description="Conditions that must be satisfied before proceeding (if any)"
    )
    further_investigation: List[str] = Field(
        description="Areas requiring deeper diligence before a final decision"
    )


# --- Agent logic ---

EXTRACTOR_SYSTEM = SystemMessage(
    """You are a due diligence analyst extracting structured findings from a single document.

Be specific and factual. Every key_finding must be a concrete, verifiable statement from
the document -- not an interpretation or inference. Every red_flag must cite specific
evidence. Do not repeat the same finding in both key_findings and red_flags."""
)

SYNTHESISER_SYSTEM = SystemMessage(
    """You are a senior M&A advisor synthesising multiple due diligence document reviews
into a unified risk register for a deal committee.

Your output must:
- Consolidate overlapping findings from different documents into single risk items
- Score each risk on BOTH severity (impact if it materialises) AND likelihood
- Source every risk item to the document(s) it came from
- Make a clear overall_assessment: proceed / proceed_with_conditions / do_not_proceed
- List only conditions and further_investigation items that are genuinely material

Do not pad the report with low-quality observations. Quality over quantity."""
)


def _extract(llm, doc_name: str, doc_text: str) -> DocumentFindings:
    extractor = EXTRACTOR_SYSTEM | llm.with_structured_output(DocumentFindings)
    return extractor.invoke(
        f"Document name: {doc_name}\n\nDocument text:\n{doc_text}"
    )


def _synthesise(llm, all_findings: list) -> DDReport:
    findings_text = "\n\n".join(
        "=== " + f.document_name + " (" + f.document_type + ") ===\n"
        "Key findings:\n" + "\n".join("- " + x for x in f.key_findings) + "\n"
        "Red flags:\n" + "\n".join("- " + x for x in f.red_flags) + "\n"
        "Questions raised:\n" + "\n".join("- " + x for x in f.questions_raised)
        for f in all_findings
    )
    synthesiser = SYNTHESISER_SYSTEM | llm.with_structured_output(DDReport)
    return synthesiser.invoke(
        "Synthesise the following per-document findings into a unified DD report:\n\n"
        + findings_text
    )


def run(documents: dict) -> DDReport:
    """
    Run commercial due diligence over multiple documents.

    Args:
        documents: Mapping of document_name -> document_text.

    Returns:
        DDReport with a unified risk register, overall assessment,
        conditions, and further investigation areas.
    """
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    all_findings = [_extract(llm, name, text) for name, text in documents.items()]
    return _synthesise(llm, all_findings)


print("Ready.")

## The scenario

We're reviewing a potential acquisition of a B2B SaaS company. We have four documents on the table: management accounts showing rapid growth but dangerously concentrated revenue, a software licensing contract with a change-of-control termination right held by the anchor customer, an executive bio flagging a prior business wind-down, and a regulatory warning letter about an uncertified medical device on the market. The agent reads each document independently, extracts every red flag, then synthesises them into a single risk register — so nothing falls through the cracks before we reach heads of terms.

In [ ]:
DOCUMENTS = {
    "Management Accounts (FY2024)": """
MERIDIAN SOFTWARE LTD -- MANAGEMENT ACCOUNTS FY2024 (UNAUDITED)

Revenue: GBP 6.8m (FY2023: GBP 4.9m, +39%)
Gross margin: 71% (FY2023: 67%)
EBITDA: GBP 0.6m (FY2023: GBP -0.2m)
Net cash position: GBP 0.4m
Monthly burn: GBP 140k

Revenue breakdown:
- Customer A (HealthGroup PLC): GBP 3.8m (56% of total revenue)
- Customer B (InsureCo Ltd): GBP 1.1m (16%)
- All other customers: GBP 1.9m (28%)

Recurring SaaS subscriptions: 74% of revenue
Implementation and professional services: 26%

Headcount: 48 FTE. Engineering: 24. Sales: 6. Support: 8. Management: 10.
Chief Product Officer departed October 2024. Role currently vacant.
Debtor days: 94 (sector benchmark: 40)
HealthGroup invoices totalling GBP 420k overdue more than 90 days.
""",
    "Software Licence Agreement (HealthGroup PLC)": """
SOFTWARE LICENCE AND SERVICES AGREEMENT
MERIDIAN SOFTWARE LTD AND HEALTHGROUP PLC

Term: 36 months commencing 1 July 2023, expiring 30 June 2026.
Annual contract value: GBP 3.8m. No auto-renewal. Renewal by mutual agreement only.

Clause 5 -- Change of Control
HealthGroup PLC may terminate this Agreement with immediate effect upon written notice
if Meridian Software Ltd undergoes a change of control event, including but not limited
to acquisition, merger, or majority shareholding transfer. No breakage fee applies.

Clause 8 -- Exclusivity
For the duration of this Agreement, Meridian shall not licence the HealthGroup Module
or any substantially similar functionality to any direct competitor of HealthGroup PLC
operating in the UK private healthcare sector.

Clause 12 -- Liability Cap
Meridian's total aggregate liability under this Agreement shall not exceed one year's
fees. However, this cap shall not apply to data protection breaches or wilful misconduct.
""",
    "Founder Biography -- Laura Mendes (CEO)": """
LAURA MENDES -- FOUNDER AND CEO

Laura founded Meridian Software in 2018 following eight years in healthcare IT.
She holds an MSc in Health Informatics and an MBA from London Business School.
Laura holds 52% of ordinary shares. Co-founder and CTO Raj Patel holds 33%.
Remaining 15% held by two seed investors.

Prior to Meridian, Laura served as Head of Product at CareTrack Systems Ltd,
which was wound down in 2016 after losing its NHS framework contract.
Laura was not a director or shareholder of CareTrack at the time of winding down.

Laura has indicated she intends to remain with the business post-acquisition
for a minimum of three years subject to agreeing satisfactory earnout terms.
""",
    "MHRA Warning Letter": """
MEDICINES AND HEALTHCARE PRODUCTS REGULATORY AGENCY
WARNING LETTER -- Ref: MHRA/2024/SW-1194
Date: 3 September 2024

Meridian Software Ltd's clinical decision support module has been identified as a
Class IIa medical device under UK MDR 2002 (as amended). Meridian has been placing
this device on the UK market without the required UKCA certification since January 2023.

Meridian is required to:
1. Cease distribution of the uncertified module within 30 days of this notice.
2. Submit a conformity assessment application to an approved UK Notified Body
   within 90 days.
3. Provide a written response to MHRA within 14 days confirming a remediation plan.

Failure to comply may result in enforcement action including product recall and
financial penalties under the Medicines Act 1968.
""",
}

report = run(DOCUMENTS)

SEV = {"critical": "CRIT", "high": "HIGH", "medium": "MED ", "low": "LOW "}
LIK = {"high": "H", "medium": "M", "low": "L"}

print("\n" + "=" * 65)
print("COMMERCIAL DUE DILIGENCE REPORT")
print("=" * 65)
print(f"Target:     {report.target_company or 'Meridian Software Ltd'}")
print(f"Assessment: {report.overall_assessment.upper()}")
print(f"\nEXECUTIVE SUMMARY\n{report.executive_summary}")

print(f"\nRISK REGISTER ({len(report.risk_items)} items)")
print(f"  {'SEV':<6} {'LIK':<4} {'AREA':<14} TITLE")
print("  " + "-" * 55)
for r in report.risk_items:
    print(f"  [{SEV[r.severity]}] [{LIK[r.likelihood]}]  {r.area:<14} {r.title}")
    print(f"         Finding:  {r.finding}")
    print(f"         Source:   {r.source_document}")
    print(f"         Action:   {r.mitigation}")
    print()

if report.key_conditions:
    print(f"KEY CONDITIONS ({len(report.key_conditions)})")
    for c in report.key_conditions:
        print(f"  * {c}")

if report.further_investigation:
    print(f"\nFURTHER INVESTIGATION ({len(report.further_investigation)})")
    for i in report.further_investigation:
        print(f"  ? {i}")

## Use your own data

Replace the `DOCUMENTS` dictionary above with:
- Your own document names as keys (e.g. `"Audited Accounts FY2023"`, `"Shareholders Agreement"`)
- The plain text of each document as the corresponding value — paste it directly or load it from a file

The agent will return a scored risk register sourced to each document, an overall go/no-go recommendation, and a prioritised list of conditions and open questions for your deal team.